In [15]:
import pandas as pd

path = r"C:\Users\DPQUAI250139\Downloads\cleaned_network_traffic.csv"
df = pd.read_csv(path)

print(df.head())
print(df.shape)

   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0                80          38308                  1                       1   
1               389            479                 11                       5   
2                88           1095                 10                       6   
3               389          15206                 17                      12   
4                88           1092                  9                       6   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                            6                            6   
1                          172                          326   
2                         3150                         3150   
3                         3452                         6660   
4                         3150                         3152   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                      6                      6            

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import tensorflow as tf
from keras import Model
from keras.layers import Input, Dense, Dropout
from keras.layers import LSTM

In [17]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 80 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  

In [18]:
current_columns = df.columns.tolist()
columns_to_keep_final = []

# Explicitly keep 'Time' and 'Label'
columns_to_keep_final.extend(['Time', 'Label'])

# Keywords for 'flows', 'bytes', 'ports' categories
# 'Length', 'Size', 'Segment' are included in bytes_keywords as they represent byte-related measurements.
flow_keywords = ['Flow', 'IAT', 'Active', 'Idle']
bytes_keywords = ['Bytes', 'Length', 'Size', 'Segment', 'Init_Win_bytes']
port_keywords = ['Port']

# Iterate through columns and add to final list if they match criteria
for col in current_columns:
    if col in columns_to_keep_final:
        continue

    if any(pk in col for pk in port_keywords):
        columns_to_keep_final.append(col)
        continue

    if any(fk in col for fk in flow_keywords):
        columns_to_keep_final.append(col)
        continue

    if any(bk in col for bk in bytes_keywords):
        columns_to_keep_final.append(col)
        continue

# Drop columns that are NOT in `columns_to_keep_final`
columns_to_drop = [col for col in current_columns if col not in columns_to_keep_final]

# Perform the drop operation
df = df.drop(columns=columns_to_drop)

print(f"Dropped {len(columns_to_drop)} columns. Remaining columns: {df.columns.tolist()}")
print(f"New DataFrame shape: {df.shape}")
df.head()

Dropped 25 columns. Remaining columns: ['Destination Port', 'Flow Duration', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', 'Bwd Avg Bytes/Bulk', 'Subflow Fwd Bytes', 'Subflow Bwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'Active Mean', 'Ac

,Destination Port,Flow Duration,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Time
0,80,38308,6,6,6,6,6.000000,0.000000,6,6,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,08:00:00
1,389,479,172,326,79,0,15.636364,31.449238,163,0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,08:00:00.133533
2,88,1095,3150,3150,1575,0,315.000000,632.561635,1575,0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,08:00:00.267067
3,389,15206,3452,6660,1313,0,203.058823,425.778474,3069,0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,08:00:00.400601
4,88,1092,3150,3152,1575,0,350.000000,694.509719,1576,0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,08:00:00.534135


In [19]:
columns_to_drop_now = [
    'Fwd Packet Length Max',
    'Fwd Packet Length Min',
    'Fwd Packet Length Mean',
    'Fwd Packet Length Std',
    'Bwd Packet Length Max',
    'Bwd Packet Length Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Std',
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Total',
    'Fwd IAT Mean',
    'Fwd IAT Std',
    'Fwd IAT Max',
    'Fwd IAT Min',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Bwd IAT Min',
    'Min Packet Length',
    'Max Packet Length',
    'Packet Length Mean',
    'Packet Length Std',
    'Packet Length Variance',
    "Idle Min",
    "Idle Max",
    "Idle Mean",
    "Idle Std",
    "Active Min",
    "Active Mean",
    "Active Max",
    "Active Std","Total Length of Fwd Packets",
"Total Length of Bwd Packets",
"Bwd IAT Total",
"Fwd Header Length",
"Bwd Header Length",
"Avg Fwd Segment Size",
"Avg Bwd Segment Size",
"Fwd Header Length","Bwd Avg Bytes/Bulk","Fwd Avg Bytes/Bulk"
]

# Drop the specified columns
df = df.drop(columns=columns_to_drop_now, errors='ignore')

print(f"Dropped {len(columns_to_drop_now)} additional columns. New DataFrame shape: {df.shape}")
print("Remaining columns:")
print(df.columns.tolist())
df.head()

Dropped 44 additional columns. New DataFrame shape: (2830743, 12)
Remaining columns:
['Destination Port', 'Flow Duration', 'Flow Bytes/s', 'Flow Packets/s', 'Average Packet Size', 'Fwd Header Length.1', 'Subflow Fwd Bytes', 'Subflow Bwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'Label', 'Time']


,Destination Port,Flow Duration,Flow Bytes/s,Flow Packets/s,Average Packet Size,Fwd Header Length.1,Subflow Fwd Bytes,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,Label,Time
0,80,38308,3.132505e+02,52.208416,9.000000,20,6,6,255,946,BENIGN,08:00:00
1,389,479,1.039666e+06,33402.922760,31.125000,368,172,326,29200,260,BENIGN,08:00:00.133533
2,88,1095,5.753425e+06,14611.872150,393.750000,336,3150,3150,29200,2081,BENIGN,08:00:00.267067
3,389,15206,6.650007e+05,1907.141918,348.689655,560,3452,6660,29200,0,BENIGN,08:00:00.400601
4,88,1092,5.771062e+06,13736.263740,420.133333,304,3150,3152,29200,2081,BENIGN,08:00:00.534135


In [20]:
df.duplicated().sum()


np.int64(0)

In [21]:
df.isnull().sum().sum()


np.int64(1358)

In [22]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer


In [23]:

# Identify numerical columns for imputation
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

# Separate numerical and non-numerical columns
df_numerical = df[numerical_cols].copy() # Explicitly create a copy
df_non_numerical = df.drop(columns=numerical_cols)


In [24]:

# Replace infinite values with NaN
df_numerical.replace([np.inf, -np.inf], np.nan, inplace=True)


In [25]:

# Initialize KNNImputer
# You can adjust n_neighbors based on your data characteristics; 5 is a common starting point.
imputer = KNNImputer(n_neighbors=5)


In [ ]:

# Perform imputation on numerical data
df_imputed_numerical = imputer.fit_transform(df_numerical)


In [ ]:

# Convert the imputed array back to a DataFrame
df_imputed_numerical = pd.DataFrame(df_imputed_numerical, columns=numerical_cols, index=df.index)

# Recombine with non-numerical columns
df = pd.concat([df_imputed_numerical, df_non_numerical], axis=1)

print("Missing values after KNN imputation:")
print(df.isnull().sum().sum())
print("DataFrame head after imputation:")

print(df.head())

Missing values after KNN imputation:
0
DataFrame head after imputation:
   Destination Port  Flow Duration  Flow Bytes/s  Flow Packets/s  \
0              80.0        38308.0  3.132505e+02       52.208416   
1             389.0          479.0  1.039666e+06    33402.922760   
2              88.0         1095.0  5.753425e+06    14611.872150   
3             389.0        15206.0  6.650007e+05     1907.141918   
4              88.0         1092.0  5.771062e+06    13736.263740   

   Average Packet Size  Fwd Header Length.1  Subflow Fwd Bytes  \
0             9.000000                 20.0                6.0   
1            31.125000                368.0              172.0   
2           393.750000                336.0             3150.0   
3           348.689655                560.0             3452.0   
4           420.133333                304.0             3150.0   

   Subflow Bwd Bytes  Init_Win_bytes_forward  Init_Win_bytes_backward   Label  \
0                6.0                   25